# paper-verdict-finetune -- Colab GPU runner

Before running: Runtime -> Change runtime type -> GPU.

This notebook is intentionally thin: it clones the repo, installs
requirements, and calls the scripts in `src/`. All logic lives there.

**Persistence model** (so a run can be launched, closed, and reviewed
later -- Colab's local disk does not survive a runtime reset):
- The model is trained from scratch (no pretrained weights), so a
  checkpoint is small -- tens of MB, not hundreds. That means it fits
  directly in git: `best_model.pt`, `vocab.json`, `config.json`,
  `train_summary.json`, and `eval_report.json` for the run all get
  committed and pushed straight back to this branch. No third-party
  storage, no Drive, no manual download step.
- `RUNS.md` is always a current, append-only index of every run and
  what it scored, readable from any future session with nothing more
  than `git clone`.

In [ ]:
!nvidia-smi

In [ ]:
import os

In [ ]:
import os

BRANCH = "verdict-finetune-architecture"
!git clone --branch $BRANCH --depth 1 https://github.com/AngryWatership/textor.git repo
%cd repo
!git config user.email "noreply@anthropic.com"
!git config user.name "Colab runner"

# anonymous clone above only grants read access -- pushing the run back
# at the end needs a GitHub token with write access to this repo.
#
# ONE-TIME SETUP (do this once per Google account, not per run):
#   1. Get a token: github.com/settings/tokens -> Generate new token
#      (classic) -> tick the "repo" scope -> Generate.
#   2. In this Colab tab, click the key icon in the LEFT sidebar
#      ("Secrets"). Add a secret named GITHUB_TOKEN, paste the token
#      as its value, and turn on "Notebook access".
# After that one-time setup, every future run of this notebook picks it
# up automatically below -- no typing, no pasting, nothing printed.
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    print("found GITHUB_TOKEN in Colab Secrets -- push step will be automatic")
except Exception:
    import getpass
    print("no GITHUB_TOKEN secret found -- see the one-time setup steps above "
          "to avoid typing this every run. Pasting it here now works too, "
          "it's just not saved for next time.")
    GITHUB_TOKEN = getpass.getpass("GitHub token (repo write access, for the push step at the end): ")

os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
del GITHUB_TOKEN

In [ ]:
!pip install -q -r requirements.txt

## Review past runs before launching a new one

`RUNS.md` came in with the clone -- it's the current index of everything
already tried. Look at this before picking the next config, so you're
not re-running something already done, and so you know what the current
best is to try to beat.

In [ ]:
import os
if os.path.exists("RUNS.md"):
    print(open("RUNS.md").read())
else:
    print("no RUNS.md yet -- this will be the first logged run")

In [ ]:
# sanity-check the split before committing GPU time to training
!python src/data.py --data data/papers.jsonl --split-out outputs/split.json

In [ ]:
import sys
sys.path.insert(0, "src")
from runlog import new_run_id

# no pretrained model anywhere -- this is a from-scratch nn.Transformer
# (src/model.py), so these pick its size/depth, not which checkpoint to load
EMBED_DIM = 256
NUM_LAYERS = 4
MODEL_TAG = f"scratch-transformer_{EMBED_DIM}d{NUM_LAYERS}l"

RUN_ID = new_run_id(MODEL_TAG, "from-scratch")
print("RUN_ID:", RUN_ID)
os.environ["RUN_ID"] = RUN_ID
os.environ["EMBED_DIM"] = str(EMBED_DIM)
os.environ["NUM_LAYERS"] = str(NUM_LAYERS)

In [ ]:
!cd src && python train.py \
    --embed-dim $EMBED_DIM \
    --num-layers $NUM_LAYERS \
    --run-id $RUN_ID \
    --data ../data/papers.jsonl \
    --split-out ../outputs/split.json \
    --out-dir ../outputs \
    --repo-root ..

In [ ]:
!cd src && python evaluate.py \
    --run-id $RUN_ID \
    --embed-dim $EMBED_DIM \
    --num-layers $NUM_LAYERS \
    --data ../data/papers.jsonl \
    --split-out ../outputs/split.json \
    --out-dir ../outputs \
    --repo-root ..

In [ ]:
import json
report = json.load(open(f"outputs/{RUN_ID}/eval_report.json"))
print("confusion matrix (rows=true, cols=pred), labels:", report["confusion_matrix_labels"])
for row in report["confusion_matrix"]:
    print(row)
print("test silhouette:", report["test_silhouette_euclidean"])
print("test knn10 purity:", report["test_knn10_purity"])
print("per-class purity:", report["test_knn10_purity_per_class"])
print("\ncompare against docs/prior_results.md and RUNS.md for every other run so far.")

## Persist: commit the full run (incl. weights) straight to GitHub

In [ ]:
from runlog import upsert_run

# checkpoint is small (random-init from-scratch model, tens of MB) and
# commits directly below -- record its path relative to the repo root
checkpoint_repo_path = f"outputs/{RUN_ID}/best_model.pt"
upsert_run(".", RUN_ID, {"checkpoint_path": checkpoint_repo_path})
print("checkpoint will be committed directly to the repo at", checkpoint_repo_path)

In [ ]:
# push the full run -- config, vocab, checkpoint, summary, eval report --
# plus the repo-level RUNS.md/runs_log.jsonl, back to the branch. The
# model is small enough (from-scratch, tens of MB) that there's no need
# to keep weights out of git the way a pretrained checkpoint would need.
!git add RUNS.md runs_log.jsonl outputs/$RUN_ID/
!du -sh outputs/$RUN_ID/best_model.pt
!git commit -m "Log run $RUN_ID" || echo "nothing new to commit"

# the clone above has no stored credentials (it was anonymous, read-only),
# so pushing needs the token entered earlier -- embed it in the URL only
# for this one push, then put the plain URL back so the token doesn't
# linger in .git/config
!git remote set-url origin https://$GITHUB_TOKEN@github.com/AngryWatership/Textor.git
!git push origin $BRANCH
!git remote set-url origin https://github.com/AngryWatership/textor.git

## To resume / review in a later session

- `git clone` this branch -> `RUNS.md` shows every run and its metrics,
  and the checkpoint itself (`outputs/<run_id>/best_model.pt`) is right
  there in the clone, no extra download step.
- To reuse a checkpoint: point `evaluate.py --checkpoint` and `--vocab`
  at `outputs/<run_id>/best_model.pt` and `outputs/<run_id>/vocab.json`
  from the clone.